# 03 · Text Similarity — Captions Across Personas

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from itertools import combinations
from scipy.stats import mannwhitneyu

from src.config import FIGURES, OUTPUTS, FONT_SCALE, SENT_COLORS, SENT_COLORS_3
from src.data_loading import load_annotations, parse_demographics, create_profiles
from src.embeddings import load_or_encode_with_ids
from src.similarity import (
    compute_per_image_similarity, compute_per_persona_similarity,
    compute_within_cross_similarity, compute_profile_coherence,
)

MUTED = sns.color_palette("muted")
sns.set_theme(style="whitegrid", font_scale=FONT_SCALE)
plt.rcParams["figure.dpi"] = 150

FIGURES.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)

df = load_annotations()
df = parse_demographics(df)
df = create_profiles(df)
print(f"Records: {len(df):,} | Images: {df['image_id'].nunique():,} | Personas: {df['persona_id'].nunique():,}")

## 1 · Encode captions with SentenceTransformer

In [ ]:
EMBED_CACHE = OUTPUTS / "caption_embeddings.npy"
ID_CACHE    = OUTPUTS / "caption_embeddings_ids.csv"

cap_embs, cap_idx = load_or_encode_with_ids(df, "caption", EMBED_CACHE, ID_CACHE)
print(f"Caption embeddings: {cap_embs.shape}")

## 2 · Per-image mean cosine similarity

In [ ]:
SIMILARITY_CACHE = OUTPUTS / "per_image_cosine_similarity.csv"

sim_df = compute_per_image_similarity(cap_embs, df, cap_idx, SIMILARITY_CACHE)
print(f"Per-image caption similarity: {len(sim_df):,} images")
sim_df.head()

## 8 · Justification text similarity (per image)

In [ ]:
JUST_EMBED_CACHE = OUTPUTS / "justification_embeddings.npy"
JUST_SIM_CACHE   = OUTPUTS / "per_image_just_similarity.csv"

just_embs, just_idx = load_or_encode_with_ids(df, "justification", JUST_EMBED_CACHE, ID_CACHE)
just_sim_df_raw = compute_per_image_similarity(just_embs, df, just_idx, JUST_SIM_CACHE)
# rename columns to match downstream expectations
just_sim_df = just_sim_df_raw.rename(columns={
    "mean_sim": "mean_just_sim",
    "std_sim":  "std_just_sim",
    "min_sim":  "min_just_sim",
    "max_sim":  "max_just_sim",
})
print(f"Justification similarity: {len(just_sim_df):,} images")
just_sim_df.head()

## 9 · Within-persona vs. cross-persona text similarity

In [ ]:
# Demographics and profiles are already parsed at load time (cell 1).
# demo_cols are available via df columns.
demo_cols = ["gender", "economic_status", "political_spectrum", "personality"]
print(f"Profile columns: {demo_cols}")
print(f"Unique profiles: {df['profile'].nunique()}")
df[demo_cols + ['profile']].head(3)

In [ ]:
WITHIN_CROSS_CACHE = OUTPUTS / "within_cross_persona_sim.csv"

wc_df = compute_within_cross_similarity(
    df, cap_embs, just_embs, cap_idx,
    cache_path=WITHIN_CROSS_CACHE,
)
print(f"Within/cross similarity: {len(wc_df):,} rows")
wc_df.head()

In [ ]:
from scipy.stats import mannwhitneyu, wilcoxon

DIM_LABELS = {
    "gender":             "Gender",
    "economic_status":    "Economic status",
    "political_spectrum": "Political spectrum",
    "personality":        "Personality",
}

MODALITY_YLABELS = {
    "caption":       "Caption\ncosine sim.",
    "justification": "Justification\ncosine sim.",
    "perception":    "Perception\nJaccard sim.",
}

fig, axes = plt.subplots(3, 4, figsize=(18, 13), sharey="row")

for row_i, modality in enumerate(["caption", "justification", "perception"]):
    mod_df = wc_df[wc_df["modality"] == modality]

    for col_i, dim in enumerate(demo_cols):
        ax = axes[row_i][col_i]
        sub = mod_df[mod_df["dimension"] == dim].dropna(subset=["within_mean", "cross_mean"])

        within_vals = sub["within_mean"].values
        cross_vals  = sub["cross_mean"].values

        data_plot = pd.DataFrame({
            "similarity": np.concatenate([within_vals, cross_vals]),
            "group": ["Within-persona"] * len(within_vals) + ["Cross-persona"] * len(cross_vals),
        })
        palette = {"Within-persona": MUTED[0], "Cross-persona": MUTED[1]}
        sns.boxplot(data=data_plot, x="group", y="similarity", hue="group",
                    palette=palette, legend=False, ax=ax,
                    order=["Within-persona", "Cross-persona"])

        # Mann-Whitney U test
        stat, p = mannwhitneyu(within_vals, cross_vals, alternative="two-sided")
        sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
        if row_i == 0:
            ax.set_title(DIM_LABELS[dim], fontsize=11)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(axis="x", rotation=15)

    axes[row_i][0].set_ylabel(MODALITY_YLABELS[modality], fontsize=10)

plt.tight_layout()
fig.savefig(FIGURES / "fig_within_cross_persona_sim.pdf", bbox_inches="tight")
fig.savefig(FIGURES / "fig_within_cross_persona_sim.png", bbox_inches="tight")
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────
_delta = "\u0394"
print(f"\n{chr(77)*0}Modality        Dimension               Within     Cross       {_delta}  p")
print("-" * 72)
for modality in ["caption", "justification", "perception"]:
    for dim in demo_cols:
        sub = wc_df[(wc_df["modality"] == modality) & (wc_df["dimension"] == dim)].dropna(
            subset=["within_mean", "cross_mean"])
        w  = sub["within_mean"].mean()
        c  = sub["cross_mean"].mean()
        stat, p = mannwhitneyu(sub["within_mean"].values, sub["cross_mean"].values, alternative="two-sided")
        sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
        print(f"{modality:14s}  {DIM_LABELS[dim]:22s}  {w:8.4f}  {c:8.4f}  {w-c:+7.4f}  {sig}")


## 11 · Within-persona profile coherence


In [ ]:
WITHIN_PROFILE_CACHE = OUTPUTS / "within_profile_coherence.csv"

wp_df = compute_profile_coherence(df, cap_embs, just_embs, cap_idx, WITHIN_PROFILE_CACHE)
print(f"Profile coherence computed for {len(wp_df)} profiles")
wp_df.head()